### Vector stores and retrievers
This video tutorial will familiarize you with LangChain's vector store and retriever abstractions. These abstractions are designed to support retrieval of data-- from (vector) databases and other sources-- for integration with LLM workflows. They are important for applications that fetch data to be reasoned over as part of model inference, as in the case of retrieval-augmented generation.

We will cover 
- Documents
- Vector stores
- Retrievers

### Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has two attributes:

- page_content: a string representing the content;
- metadata: a dict containing arbitrary metadata.
The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual Document object often represents a chunk of a larger document.

Let's generate some sample documents:

In [2]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key=os.getenv("GROQ_API_KEY")

os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

llm=ChatGroq(groq_api_key=groq_api_key,model="llama-3.1-8b-instant")


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [5]:
from langchain_chroma import Chroma

vectorStore = Chroma.from_documents(documents=documents, embedding=embedding)   

In [6]:
vectorStore.similarity_search("cat")

[Document(id='ace5e96e-b23d-4c38-aea4-a4b87dff40b3', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='c26b0bca-ef85-4a17-97cc-843284f75de6', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='7c8737c7-490e-4834-bed4-6a920d08dc13', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='763d900a-329b-48a9-a26c-f658e2f8341a', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [7]:
## Async Query
await vectorStore.asimilarity_search("cat")

[Document(id='ace5e96e-b23d-4c38-aea4-a4b87dff40b3', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='c26b0bca-ef85-4a17-97cc-843284f75de6', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='7c8737c7-490e-4834-bed4-6a920d08dc13', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='763d900a-329b-48a9-a26c-f658e2f8341a', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

### Retrievers
LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method:

In [8]:
from typing import List
from langchain_classic.chains import retrieval
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorStore.similarity_search).bind(k=2) ## k is the top results to return
retriever.batch(["Cats","Dogs"])

[[Document(id='ace5e96e-b23d-4c38-aea4-a4b87dff40b3', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  Document(id='7c8737c7-490e-4834-bed4-6a920d08dc13', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')],
 [Document(id='c26b0bca-ef85-4a17-97cc-843284f75de6', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  Document(id='ace5e96e-b23d-4c38-aea4-a4b87dff40b3', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')]]

Vectorstores implement an as_retriever method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following:

In [ ]:

retriverFn = vectorStore.as_retriever(
    search_type="similarity", ## vectorStore.similarity_search
    search_kwargs={"k": 2}
)

retriverFn.batch(["Cats"])

[[Document(id='ace5e96e-b23d-4c38-aea4-a4b87dff40b3', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  Document(id='7c8737c7-490e-4834-bed4-6a920d08dc13', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]]

In [12]:
## integrating the retriver with the chain 

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([("human", message)])

## Creates a parallel mapping (RunnableParallel)
# What it does:
# 	•	Takes 1 input (the question string)
# 	•	Runs 2 things in parallel on that same input:
# 	•	 retriverFn(question)  → gets relevant documents → labels result as  "context" 
## Essentially we call the retriverFn on the question to get the relevant context, and we also pass the question through unchanged. Both of these run in parallel and their results are labeled accordingly for the prompt.
# 	•	 RunnablePassthrough()(question)  → returns question unchanged → labels as  "question" 
 
rag_chain={"context":retriverFn,"question":RunnablePassthrough()}|prompt|llm

# Part 2:  | prompt 
# Sequential pipe to ChatPromptTemplate
# What it does:
# 	•	Takes the dict  {"context": ..., "question": ...} 
# 	•	Fills the prompt template placeholders:
# 	•	 {question}  ←  "tell me about cats" 
# 	•	 {context}  ← document contents
# 	•	Returns formatted  ChatPromptValue  (ready for LLM)

response=rag_chain.invoke("tell me about dogs")
print(response.content)

Dogs are great companions, known for their loyalty and friendliness.
